In [1]:
!pip install vllm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.4/436.4 MB 114.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 305.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 40.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.1/888.1 MB 161.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 181.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 184.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 MB 188.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 202.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 44.9 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 114.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 31.8 MB/s eta

In [1]:
import multiprocessing as mp
mp.set_start_method("spawn", force=True)  # do this ONCE per kernel before using mp

from vllm import SamplingParams
from vllm_generate5 import build_llm, generate

model_name = "/workspace/qwen7b"
#model_name = "/workspace/llama3b-rm"
#model_name = "/workspace/llama3b-rm-converted-model"
device_groups = [[0]]  # 2 workers on GPU 0 and 1

model = build_llm(
    model_name=model_name,
    tensor_parallel_size=len(device_groups[0]),
    num_instances=len(device_groups),
    device_groups=device_groups,
    max_model_len=2500,
    max_num_seqs=64,
    gpu_memory_utilization=0.90,
)

try:
    #sp = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=32)
    prompts = ["Hello, my name is", "The capital of France is"] * 200
    outs = generate(model, prompts, batch_size=4, timeout_s=180)
    print("finished")
    print(outs[:3])

    #outs = generate(model, prompts, batch_size=4, timeout_s=180)
    #for i, (inp, out) in enumerate(zip(prompts, outs)):
    #    print(f"[#{i}] {inp!r} -> {out!r}")
finally:
    model.close()

== Worker logs ==
[Worker 0] generating: 99/100 (99.0%) · 1.24 batch/s · ETA 0.8s
finished
[' Kellie and I am a 3rd grade teacher. I have a great need for a sensory table in my classroom. This table will be used for', ' Paris. It is also the largest city in France. It is located in the north central part of the country. The city sits on the Seine River.', ' Alex and I’m here to talk to you about a topic that’s important to me: mental health. Many of us have experienced some form of mental health issues']
